In [1]:
!pip install langgraph langchain-ollama langchain-community ddgs fastapi uvicorn nest_asyncio requests

In [2]:
# 1. Install zstd dependency first, then install Ollama binary
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start Ollama server in the background and log output
import subprocess
import time

print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server a few seconds to boot up
time.sleep(5)

# 3. Pull the llama3 model
print("Pulling Llama3 model (this may take a couple of minutes)...")
!ollama pull llama3
print("Ollama is ready!")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (1,927 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [3]:
import json
import threading
import nest_asyncio
import uvicorn
from typing import TypedDict, List
from fastapi import FastAPI
from pydantic import BaseModel
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.messages import HumanMessage, SystemMessage

# Allow async loops to run nested in Colab
nest_asyncio.apply()

# ==========================================
# 1. INIT: Free Local Model and Free Search
# ==========================================
llm = ChatOllama(model="llama3", temperature=0, format="json")
markdown_llm = ChatOllama(model="llama3", temperature=0.2)
search_tool = DuckDuckGoSearchResults(max_results=3)

# ==========================================
# 2. STATE: Shared Agent Memory
# ==========================================
class ResearchState(TypedDict):
    original_prompt: str
    search_queries: List[str]
    raw_data: List[str]
    draft_report: str
    critic_feedback: str
    iteration_count: int

# ==========================================
# 3. ORG CHART: The Agents (Nodes)
# ==========================================
def planner_agent(state: ResearchState):
    prompt = state["original_prompt"]
    system_msg = SystemMessage(content=(
        "You are a research planner. Break the user's request down into exactly 3 highly specific search engine queries. "
        "You must respond ONLY with a JSON object containing a list of strings under the key 'queries'. "
        "Example: {\"queries\": [\"query 1\", \"query 2\"]}"
    ))
    user_msg = HumanMessage(content=f"Request: {prompt}")
    response = llm.invoke([system_msg, user_msg])
    try:
        data = json.loads(response.content)
        queries = data.get("queries", [prompt])
    except:
        queries = [prompt]
    return {"search_queries": queries, "iteration_count": state.get("iteration_count", 0)}

def researcher_agent(state: ResearchState):
    queries = state["search_queries"]
    feedback = state.get("critic_feedback", "")

    if feedback and feedback != "PASS":
        refine_msg = SystemMessage(content=(
            "You are a researcher. Based on the Critic's feedback, generate 2 new search queries to fill the knowledge gaps. "
            "Respond ONLY with a JSON object containing a list of strings under the key 'queries'."
        ))
        user_msg = HumanMessage(content=f"Original Queries: {queries}\nCritic Feedback: {feedback}")
        response = llm.invoke([refine_msg, user_msg])
        try:
            data = json.loads(response.content)
            queries = data.get("queries", queries)
        except:
            pass

    new_raw_data = []
    for query in queries:
        try:
            results = search_tool.invoke({"query": query})
            new_raw_data.append(f"Source [DuckDuckGo - {query}]:\n{results}")
        except Exception as e:
            new_raw_data.append(f"Search failed for {query}: {str(e)}")

    current_data = state.get("raw_data", [])
    return {"raw_data": current_data + new_raw_data}

def synthesizer_agent(state: ResearchState):
    data_context = "\n\n".join(state.get("raw_data", []))
    prompt = state["original_prompt"]
    system_msg = SystemMessage(content="You are an expert synthesizer. Write a highly structured, professional markdown report answering the user's prompt. You MUST rely exclusively on the provided raw data. Do not hallucinate external facts.")
    user_msg = HumanMessage(content=f"User Prompt: {prompt}\n\nRaw Data Context:\n{data_context}")
    response = markdown_llm.invoke([system_msg, user_msg])
    return {"draft_report": response.content}

def critic_agent(state: ResearchState):
    draft = state.get("draft_report", "")
    prompt = state["original_prompt"]
    iterations = state.get("iteration_count", 0)

    if iterations >= 2:
        return {"critic_feedback": "PASS", "iteration_count": iterations + 1}

    system_msg = SystemMessage(content=(
        "You are a strict QA Reviewer. Review the draft report against the user prompt. "
        "You must respond ONLY with a JSON object. "
        "If the report is deeply detailed and answers everything, set the key 'status' to 'PASS' and 'feedback' to ''. "
        "If it is shallow or missing key info, set 'status' to 'REVISE' and provide a 1-sentence direction in the 'feedback' key. "
        "Example: {\"status\": \"REVISE\", \"feedback\": \"Missing specific features summary.\"}"
    ))
    user_msg = HumanMessage(content=f"Prompt: {prompt}\n\nDraft Report:\n{draft}")
    response = llm.invoke([system_msg, user_msg])
    try:
        data = json.loads(response.content)
        feedback = "PASS" if data.get("status") == "PASS" else data.get("feedback", "Provide more depth.")
    except:
        feedback = "PASS"
    return {"critic_feedback": feedback, "iteration_count": iterations + 1}

# ==========================================
# 4. ORCHESTRATION: Wiring the Graph
# ==========================================
workflow = StateGraph(ResearchState)
workflow.add_node("planner", planner_agent)
workflow.add_node("researcher", researcher_agent)
workflow.add_node("synthesizer", synthesizer_agent)
workflow.add_node("critic", critic_agent)

def route_critique(state: ResearchState):
    if state["critic_feedback"] == "PASS":
        return "approve"
    return "revise"

workflow.set_entry_point("planner")
workflow.add_edge("planner", "researcher")
workflow.add_edge("researcher", "synthesizer")
workflow.add_edge("synthesizer", "critic")
workflow.add_conditional_edges("critic", route_critique, {"revise": "researcher", "approve": END})

agentic_org = workflow.compile()

# ==========================================
# 5. ENTERPRISE LAYER: FastAPI Server Setup
# ==========================================
app = FastAPI(title="Nth AI - Free Agentic Research Desk")

class ResearchRequest(BaseModel):
    topic: str

@app.post("/api/v1/research")
async def run_research_desk(request: ResearchRequest):
    initial_state = {
        "original_prompt": request.topic,
        "iteration_count": 0,
        "raw_data": [],
        "search_queries": [],
        "critic_feedback": "",
        "draft_report": ""
    }
    result_state = agentic_org.invoke(initial_state)
    return {
        "status": "success",
        "iterations_required": result_state["iteration_count"],
        "final_report": result_state["draft_report"]
    }

# Function to run the server in a thread
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Start server thread if not already running
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("FastAPI server is running in the background on http://127.0.0.1:8000")

/tmp/ipykernel_17127/2204242976.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


FastAPI server is running in the background on http://127.0.0.1:8000


In [5]:
import requests
from IPython.display import display, Markdown

url = "http://127.0.0.1:8000/api/v1/research"
payload = {"topic": "What are the latest performance advancements in Llama 3 architectures?"}

print("Sending request to your agentic organization... (This will take a moment as agents execute self-correction loops)")
response = requests.post(url, json=payload)

if response.status_code == 200:
    res_data = response.json()
    print(f"\nSuccess! Total internal agent loops/iterations: {res_data['iterations_required']}")
    print("-" * 50)
    display(Markdown(res_data["final_report"]))
else:
    print("Error:", response.text)

Sending request to your agentic organization... (This will take a moment as agents execute self-correction loops)
INFO:     127.0.0.1:58648 - "POST /api/v1/research HTTP/1.1" 200 OK

Success! Total internal agent loops/iterations: 3
--------------------------------------------------


**Latest Performance Advancements in LLaMA 3 Architectures**

The provided raw data does not explicitly mention specific numerical improvements achieved through advancements in LLaMA 3 architectures. However, it provides insights into the performance enhancements and key differences between LLaMA 2 and LLaMA 3.

**Key Differences: Architecture & Training**

According to [DuckDuckGo - metrics for evaluating language model advancements in LLaMA 3], the main difference between LLaMA 2 and LLaMA 3 is their architecture and training. The 8B version of LLaMA 3 was nearly as powerful as the largest LLaMA 2, with the 70B model still learning even at the end of the 15T tokens training.

**Performance Enhancements**

The data suggests that LLaMA 3 has shown performance enhancements in various areas. For instance, [DuckDuckGo - What specific metrics were used to measure the performance enhancements in recent studies?] mentions that fine-tuned LLM-based metrics perform well overall, outperforming other models in multiple dimensions.

**Evaluation Metrics**

The provided data highlights several evaluation metrics used to assess the performance of LLaMA 3 architectures. These include:

1. MoverScore (uses Earth Mover's Distance on embeddings)
2. Embedding from GPT-3
3. Feedback accuracy, relevance, localization, and rating consistency

These metrics aim to capture the semantic overlap or adequacy of generated text relative to a reference or source.

**Conclusion**

While the provided data does not provide specific numerical improvements achieved through advancements in LLaMA 3 architectures, it highlights key differences between LLaMA 2 and LLaMA 3, as well as performance enhancements and evaluation metrics used to assess their performance.